In [1]:
import pandas as pd
from h3 import h3
import glob
import numpy as np
from pandas import merge
import json
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import shape
from datetime import datetime, timedelta
import seaborn as sns

In [7]:
# Pandas Config
pd.set_option('display.max_columns', None)
pd.options.display.max_rows = 200

# ShareNow Data 
VEHICLE_TRIP = '/Users/demircanm/sciebo/IS3/Research_Projects/EmpiricalAnalysisSharedMobility/02_Data/final_dataset.csv'

# TIMEZONE
TIMEZONE = 'Europe/Berlin'

In [8]:
trips = pd.read_csv(VEHICLE_TRIP)

/Users/demircanm/opt/anaconda3/envs/SharedMobility/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3146: DtypeWarning: Columns (4,21,22,114) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [9]:
# Calc H3 Start and End Locations
import h3._cy as c

# Resolution 7
trips["h3_start_7"] = trips.apply(lambda row: c.geo_to_h3(row["latitude_start"], row["longitude_start"], 7), axis = 1)
trips["h3_end_7"] = trips.apply(lambda row: c.geo_to_h3(row["latitude_end"], row["longitude_end"], 7), axis = 1)

# Resolution 8
trips["h3_start_8"] = trips.apply(lambda row: c.geo_to_h3(row["latitude_start"], row["longitude_start"], 8), axis = 1)
trips["h3_end_8"] = trips.apply(lambda row: c.geo_to_h3(row["latitude_end"], row["longitude_end"], 8), axis = 1)

In [10]:
# Convert to H3 Hex ID
import h3._cy as c

# Hex 7
trips["h3_start_7"] = trips["h3_start_7"].apply(lambda row: c.int2hex(row))
trips["h3_end_7"] = trips["h3_end_7"].apply(lambda row: c.int2hex(row))

# Hex 8
trips["h3_start_8"] = trips["h3_start_8"].apply(lambda row: c.int2hex(row))
trips["h3_end_8"] = trips["h3_end_8"].apply(lambda row: c.int2hex(row))

# Hex 9 
trips.rename(columns={"hex_start":"h3_start_9", "hex_end":"h3_end_9"}, inplace = True)

In [16]:
# Replace wrong vehicle type
trips["vehicleType"] = trips["vehicleType"].str.replace("bycicle", "bicycle")

In [26]:
# Get Trip Count per Hexagon per Vehicle Type
trips_cnt = trips.groupby(["h3_start_7", "vehicleType"]).agg({"id":"count"})

In [38]:
# Unstack 
unstacked = trips_cnt.unstack()

In [39]:
# Check Column order
unstacked.columns

MultiIndex([('id',      'bicycle'),
            ('id',          'car'),
            ('id', 'kick scooter'),
            ('id',      'scooter')],
           names=[None, 'vehicleType'])

In [40]:
# Rename columns
unstacked.columns = ['bikes', 'cars', 'kick_scooters', 'scooters']

In [43]:
# Replace NaN values
unstacked.fillna(0, inplace = True)

In [48]:
unstacked["bikes_prob"] = unstacked["bikes"]/unstacked["bikes"].sum()
unstacked["cars_prob"] = unstacked["cars"]/unstacked["cars"].sum()
unstacked["kick_scooters_prob"] = unstacked["kick_scooters"]/unstacked["kick_scooters"].sum()
unstacked["scooters_prob"] = unstacked["scooters"]/unstacked["scooters"].sum()

In [98]:
unstacked = unstacked.reset_index()

In [99]:
# Sample Supply Distribution for Cars
N_CARS = 1000

cars_prob_arr = unstacked["cars_prob"].to_numpy()

car_cnt = pd.DataFrame(np.random.multinomial(N_CARS, cars_prob_arr, size=1).transpose(), columns = ["car_count"])

car_cnt.index = unstacked.index

unstacked["car_cnt"] = car_cnt["car_count"]

In [100]:
# Sample Supply Distribution for Cars
N_SCOOTERS = 500

scooters_prob_arr = unstacked["scooters_prob"].to_numpy()

scooters_cnt = pd.DataFrame(np.random.multinomial(N_SCOOTERS, scooters_prob_arr, size=1).transpose(), columns = ["scooters_count"])

scooters_cnt.index = unstacked.index

unstacked["scooters_cnt"] = scooters_cnt["scooters_count"]

In [101]:
# Sample Supply Distribution for Cars
N_KICK_SCOOTERS = 1000

kick_scooters_prob_arr = unstacked["kick_scooters_prob"].to_numpy()

kick_scooters_cnt = pd.DataFrame(np.random.multinomial(N_KICK_SCOOTERS, kick_scooters_prob_arr, size=1).transpose(), columns = ["kick_scooters_count"])

kick_scooters_cnt.index = unstacked.index

unstacked["kick_scooters_cnt"] = kick_scooters_cnt["kick_scooters_count"]


In [102]:
# Sample Supply Distribution for Cars
N_BIKES = 1000

bikes_prob_arr = unstacked["bikes_prob"].to_numpy()

bikes_cnt = pd.DataFrame(np.random.multinomial(N_BIKES, bikes_prob_arr, size=1).transpose(), columns = ["bikes_count"])

bikes_cnt.index = unstacked.index

unstacked["bikes_cnt"] = bikes_cnt["bikes_count"]


In [103]:
unstacked[["h3_start_7", "car_cnt", "scooters_cnt", "kick_scooters_cnt", "bikes_cnt"]]

,h3_start_7,car_cnt,scooters_cnt,kick_scooters_cnt,bikes_cnt
0,871f18840ffffff,0,0,0,0
1,871f18841ffffff,0,0,0,0
2,871f18844ffffff,0,0,0,0
3,871f18845ffffff,0,0,0,0
4,871f18854ffffff,0,0,0,0
5,871f18855ffffff,0,0,0,0
6,871f18866ffffff,0,0,0,0
7,871f18868ffffff,0,0,0,0
8,871f18869ffffff,7,0,0,0
9,871f1886bffffff,3,0,0,0


In [105]:
# Save Probs

unstacked.rename(columns={"h3_start_7":"hex_id_7"}, inplace=True)

In [108]:
result = unstacked[["hex_id_7", "cars_prob", "scooters_prob", "kick_scooters_prob", "bikes_prob"]]

In [109]:
result.to_csv("vehicle_supply_distribution_hex_7.csv", index=False)